In [ ]:
import torch
from PIL import Image
import pandas as pd
import ast
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import os
import matplotlib.pyplot as plt



class FrameSequenceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.data = df
        self.transform = transform

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        frame_paths = row['frame_paths']  # Lista de rutas
        label = int(row['label'])
        coeficient = row['coeficient_Amet']

        prev_frames = []
        for frame_path in frame_paths[:-1]:  # frames previos
            image = Image.open(frame_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            prev_frames.append(image)

        chute_frame = Image.open(frame_paths[-1]).convert('RGB')
        if self.transform:
            chute_frame = self.transform(chute_frame)

        prev_frames = torch.stack(prev_frames)  # Tensor (N, C, H, W)

        return prev_frames, chute_frame, label, coeficient



In [ ]:
import os
import pandas as pd

def create_dataframe_for_frame_sequence_with_coef(dataset_root):
    rows = []

    for shot_folder in os.listdir(dataset_root):
        folder_path = os.path.join(dataset_root, shot_folder)
        if not os.path.isdir(folder_path):
            continue

        label_path = os.path.join(folder_path, "label.txt")
        if not os.path.exists(label_path):
            print(f"Etiqueta no encontrada en {folder_path}")
            continue

        try:
            with open(label_path, "r") as f:
                lines = f.readlines()
                label = int(lines[0].strip())  # La primera línea es la etiqueta

                # Buscamos coeficient_AMET en las siguientes líneas
                coeficient_AMET = None
                for line in lines[1:]:
                    if 'frame_shot.jpg' in line:
                        coeficient_AMET = float(line.split(':')[-1].strip())
                        break

                if coeficient_AMET is None:
                    print(f"No se encontró coeficient_AMET en {folder_path}")
                    continue
        except ValueError:
            print(f"Etiqueta o coeficiente inválido en {folder_path}")
            continue

        frame_paths = []
        for i in range(5):  # Los 5 frames previos
            prev_frame_path = os.path.join(folder_path, f"frame_prev_{i}.jpg")
            if os.path.exists(prev_frame_path):
                frame_paths.append(prev_frame_path)

        shot_frame_path = os.path.join(folder_path, "frame_shot.jpg")
        if os.path.exists(shot_frame_path):
            frame_paths.append(shot_frame_path)
        else:
            print(f"Frame del chute no encontrado en {folder_path}")
            continue

        rows.append([frame_paths, label, coeficient_AMET])

    df = pd.DataFrame(rows, columns=["frame_paths", "label", "coeficient_Amet"])
    return df

dataset_path = "./Dataset_final/Frames_Bons_Definitius"
df = create_dataframe_for_frame_sequence_with_coef(dataset_path)

obj = FrameSequenceDataset(df)
print(obj)


True


In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
image_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
}

def get_dataloaders(csv_path, batch_size=32, split_ratio=0.8):
    dataset = FrameSequenceDataset(csv_path, transform=image_transforms['train'])

    train_size = int(split_ratio * len(dataset))
    val_size = len(dataset) - train_size

    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    # Replace transforms for validation set
    val_dataset.dataset.transform = image_transforms['val']

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader


In [4]:
import torch.nn as nn
import torch.nn.functional as F

class CNNLSTMClassifier(nn.Module):
    def __init__(self, cnn_out_dim=128, lstm_hidden=64):
        super(CNNLSTMClassifier, self).__init__()

        # CNN to extract features
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),  # (B, 3, 224, 224)
            nn.ReLU(),
            nn.MaxPool2d(2),                # (B, 16, 112, 112)
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                # (B, 32, 56, 56)
            nn.Flatten(),                   # (B, 32*56*56)
            nn.Linear(32 * 56 * 56, cnn_out_dim),
            nn.ReLU()
        )

        # LSTM for previous frame features
        self.lstm = nn.LSTM(input_size=cnn_out_dim, hidden_size=lstm_hidden, batch_first=True)

        # Final classification
        self.classifier = nn.Sequential(
        nn.Linear(lstm_hidden + cnn_out_dim, 64),
        nn.ReLU(),
        nn.Dropout(0.5), 
        nn.Linear(64, 1),
        nn.Sigmoid()
    )

    def forward(self, prev_frames, chute_frame):
        B, N, C, H, W = prev_frames.shape

        # Flatten temporal dimension to process through CNN
        prev_frames = prev_frames.view(B * N, C, H, W)
        prev_features = self.cnn(prev_frames)  # (B*N, cnn_out_dim)
        prev_features = prev_features.view(B, N, -1)  # (B, N, cnn_out_dim)

        # Process with LSTM
        _, (h_n, _) = self.lstm(prev_features)  # h_n: (1, B, lstm_hidden)
        lstm_out = h_n.squeeze(0)               # (B, lstm_hidden)

        # CNN feature for chute frame
        chute_feature = self.cnn(chute_frame)   # (B, cnn_out_dim)

        # Concatenate LSTM + chute
        combined = torch.cat([lstm_out, chute_feature], dim=1)  # (B, lstm_hidden + cnn_out_dim)
        output = self.classifier(combined)
        return output


In [ ]:
class CoordConvLayer(nn.Module):
   def __init__(self, with_r=False):
       super(CoordConvLayer, self).__init__()
       self.with_r = with_r


   def forward(self, x):
       """
       Añade canales coordenados a la imagen.
       x: batch x channels x height x width
       retorna: batch x (channels + 2 or 3) x height x width
       """
       batch_size, _, height, width = x.size()
       device = x.device


       xx_channel = torch.linspace(-1, 1, steps=width, device=device).repeat(1, height, 1)
       yy_channel = torch.linspace(-1, 1, steps=height, device=device).repeat(1, width, 1).transpose(1, 2)


       xx_channel = xx_channel.expand(batch_size, 1, height, width)
       yy_channel = yy_channel.expand(batch_size, 1, height, width)


       ret = torch.cat([x, xx_channel, yy_channel], dim=1)


       if self.with_r:
           rr = torch.sqrt(xx_channel ** 2 + yy_channel ** 2)
           ret = torch.cat([ret, rr], dim=1)


       return ret



class CNNLSTMClassifierWithCoordConv(nn.Module):
    def __init__(self, cnn_out_dim=128, lstm_hidden=64, with_r=False):
        super(CNNLSTMClassifierWithCoordConv, self).__init__()

        # CoordConv layer, con opción de incluir el canal r
        self.coordconv = CoordConvLayer(with_r=with_r)

        # CNN para extraer features (ahora recibirá más canales por coordconv)
        in_channels = 3 + 2 + (1 if with_r else 0)  # canales RGB + 2 coordenadas + r opcional

        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1),  # entrada con canales aumentados
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, cnn_out_dim),
            nn.ReLU()
        )

        # LSTM para características de frames previos
        self.lstm = nn.LSTM(input_size=cnn_out_dim, hidden_size=lstm_hidden, batch_first=True)

        # Clasificador final
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden + cnn_out_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, prev_frames, chute_frame,coeficient):
        B, N, C, H, W = prev_frames.shape

        # Aplicar coordconv a frames previos
        prev_frames = prev_frames.view(B * N, C, H, W)
        prev_frames = self.coordconv(prev_frames)  # Ahora con canales coordenados
        prev_features = self.cnn(prev_frames)      # (B*N, cnn_out_dim)
        prev_features = prev_features.view(B, N, -1)  # (B, N, cnn_out_dim)

        # LSTM sobre características previas
        _, (h_n, _) = self.lstm(prev_features)
        lstm_out = h_n.squeeze(0)

        # Aplicar coordconv al frame chute
        chute_frame = self.coordconv(chute_frame)
        chute_feature = self.cnn(chute_frame)

        # Concatenar salida LSTM con features chute
        coeficient=coeficient.view(-1,1)
        combined = torch.cat([lstm_out, chute_feature,coeficient], dim=1)
        output = self.classifier(combined)

        return output



In [6]:
class WeightedLossLSTM(nn.Module):
    def __init__(self, image_size, sigma=1.0):
        super(WeightedLossLSTM, self).__init__()
        self.image_size = image_size
        self.sigma = sigma
        self.register_buffer('gaussian_mask', self.create_gaussian_mask())

    def create_gaussian_mask(self):
        x = torch.arange(0, self.image_size[1]).float()
        y = torch.arange(0, self.image_size[0]).float()
        xx, yy = torch.meshgrid(y, x, indexing='ij')
        cx, cy = self.image_size[1] // 2, self.image_size[0] // 2
        distance = (xx - cx)**2 + (yy - cy)**2
        gaussian = torch.exp(-distance / (2 * self.sigma ** 2))
        return gaussian / gaussian.sum()

    def forward(self, output, target, chute_frame):
        """
        output: [B, 1] logits (from model)
        target: [B, 1] ground truth labels (float)
        chute_frame: [B, C, H, W] — frame del chute
        """

        B = output.shape[0]
        with torch.no_grad():
            # Convert to grayscale
            grayscale = chute_frame[:, :3].mean(dim=1)  # [B, H, W]

            # Apply gaussian mask
            importance = (grayscale * self.gaussian_mask).view(B, -1).sum(dim=1)  # [B]
            importance = importance / importance.sum()  # Normalize weights

        # BCE loss per sample (no reduction)
        bce = F.binary_cross_entropy_with_logits(output.view(-1), target.view(-1), reduction='none')  # [B]

        # Weighted loss
        weighted_loss = (importance * bce).sum()

        return weighted_loss


In [ ]:
def evaluate_cnn_lstm(model, val_loader, device, criterion):
    model.eval()
    correct = 0
    total = 0
    val_loss = 0.0
    expected_goal_all=[]
    with torch.no_grad():
        for prev_frames, chute_frame, labels, coeficient in val_loader:
            prev_frames = prev_frames.to(device)
            chute_frame = chute_frame.to(device)
            coeficient = coeficient.to(device).float()
            #labels = labels.to(device).float()
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(prev_frames, chute_frame, coeficient)
            expected_goal=outputs.view(-1)
            expected_goal_all.append((expected_goal.cpu(), [prev_frames, chute_frame]))

            #outputs=outputs.view(-1)
            loss=criterion(outputs,labels)
            val_loss += loss.item()*prev_frames.size(0)
            preds = (outputs > 0.5).int().squeeze()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss=val_loss/len(val_loader.dataset)
    accuracy= 100 * correct / (total*32)
    return accuracy, avg_loss


def train_cnn_lstm(model, train_loader, val_loader, epochs=50, lr=1e-5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for prev_frames, chute_frame, labels, coeficient in train_loader:
            prev_frames = prev_frames.to(device)
            chute_frame = chute_frame.to(device)
            coeficient = coeficient.to(device).float()
            labels = labels.float().unsqueeze(1).to(device)

            optimizer.zero_grad()
            outputs = model(prev_frames, chute_frame)
            #outputs=outputs.view(-1)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * prev_frames.size(0)

        val_acc, val_loss = evaluate_cnn_lstm(model, val_loader, device,criterion)
        print(f"Epoch {epoch+1}, Training_Loss: {running_loss/len(train_loader.dataset):.4f},Validation_Loss:{val_loss:.4f} ,Val Acc: {val_acc:.2f}%")

    return model


In [ ]:
def evaluate_cnn_lstm(model, val_loader, device, criterion):
    model.eval()
    correct = 0
    total = 0
    val_loss = 0.0
    expected_goal_all=[]
    with torch.no_grad():
        for prev_frames, chute_frame, labels, coeficiente in val_loader:
            prev_frames = prev_frames.to(device)
            chute_frame = chute_frame.to(device)
            coeficiente=coeficiente.to(device).float()
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(prev_frames, chute_frame,coeficiente)
            expected_goal=outputs.view(-1)
            expected_goal_all.append((expected_goal.cpu(), [prev_frames, chute_frame]))
            loss = criterion(outputs, labels)
            val_loss += loss.item() * prev_frames.size(0)
            preds = (outputs > 0.5).int().squeeze()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = val_loss / len(val_loader.dataset)
    accuracy = 100 * correct / (total*32)
    return accuracy, avg_loss


def train_cnn_lstm(model, train_loader, val_loader, epochs=30, lr=1e-5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    criterion = nn.BCELoss()
    #criterion=WeightedLossLSTM((224,224)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    val_losses = []
    val_accuracies = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for prev_frames, chute_frame, labels, coeficiente in train_loader:
            prev_frames = prev_frames.to(device)
            chute_frame = chute_frame.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            coeficiente=coeficiente.to(device).float()

            optimizer.zero_grad()
            outputs = model(prev_frames, chute_frame,coeficiente)
            loss = criterion(outputs, labels)
            #loss = criterion(outputs, labels, chute_frame)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * prev_frames.size(0)

        avg_train_loss = running_loss / len(train_loader.dataset)
        val_acc, val_loss = evaluate_cnn_lstm(model, val_loader, device, criterion)

        train_losses.append(avg_train_loss)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch {epoch+1}, Training_Loss: {avg_train_loss:.4f}, Validation_Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

    return model, train_losses, val_losses, val_accuracies






In [ ]:
import wandb
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
torch.manual_seed(0)
train_loader, val_loader = get_dataloaders(df, batch_size=32)
model=CNNLSTMClassifierWithCoordConv(cnn_out_dim=128, lstm_hidden=64)
model, train_losses, val_losses, val_accuracies = train_cnn_lstm(model, train_loader, val_loader)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

wandb.log({})

In [ ]:
plt.plot(losses["expected_goal"], label="expected_goal")
plt.xlabel("Epochs")
plt.plot
plt.legend()
plt.pause(0.000001)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

def plot_images_with_goals(expected_goal_all, n=6):
    fig, axes = plt.subplots(1, n, figsize=(15, 5))
    count = 0
    for eg_batch, img_batch in expected_goal_all:
        for eg, img in zip(eg_batch, img_batch):
            if count >= n:
                break
            img_np = img.cpu().permute(1, 2, 0).numpy()  # ← FIX AQUÍ
            axes[count].imshow(img_np)
            axes[count].set_title(f"Expected goal: {eg.item():.3f}")
            axes[count].axis('off')
            count += 1
        if count >= n:
            break
    plt.tight_layout()
    plt.show()

plot_images_with_goals(expected_goals, n=6)

NameError: name 'expected_goals' is not defined